<a href="https://colab.research.google.com/github/sam-wahid/vlm-llm-segmentation/blob/main/dinov1benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q torch torchvision
!pip install -q scikit-learn tqdm matplotlib

In [ ]:
import time
import torch
import numpy as np

from tqdm import tqdm

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cpu


In [ ]:
dino_model = torch.hub.load(
    'facebookresearch/dino:main',
    'dino_vits16'
)

dino_model = dino_model.to(device)
dino_model.eval()

print("DINOv1 Loaded")

Downloading: "https://github.com/facebookresearch/dino/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dino_deitsmall16_pretrain.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 245MB/s]


DINOv1 Loaded


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor()
])

dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

loader = DataLoader(dataset, batch_size=32, shuffle=False)

print("Total Images:", len(dataset))
print("Classes:", dataset.classes)

100%|██████████| 26.4M/26.4M [00:01<00:00, 17.1MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 272kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.03MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 19.8MB/s]

Total Images: 10000
Classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


In [ ]:
def benchmark_dino(model, loader):

    embeddings = []
    labels_all = []

    total_time = 0
    total_images = 0

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    with torch.no_grad():

        for images, labels in tqdm(loader):

            images = images.to(device)

            start = time.time()

            features = model(images)

            end = time.time()

            total_time += (end - start)
            total_images += images.size(0)

            embeddings.append(features.cpu().numpy())
            labels_all.append(labels.numpy())

    embeddings = np.concatenate(embeddings, axis=0)
    labels_all = np.concatenate(labels_all, axis=0)

    fps = total_images / total_time
    latency = total_time / total_images

    gpu_memory = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0

    return {
        "embeddings": embeddings,
        "labels": labels_all,
        "fps": fps,
        "latency": latency,
        "gpu_memory": gpu_memory
    }

In [ ]:
dino_results = benchmark_dino(dino_model, loader)

 45%|████▌     | 142/313 [16:53<20:10,  7.08s/it]

In [ ]:
X = dino_results["embeddings"]
y = dino_results["labels"]

split = int(0.8 * len(X))

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

pred = knn.predict(X_test)

acc = accuracy_score(y_test, pred)

In [ ]:
print("\n===== DINOv1 RESULTS =====")

print("Accuracy      :", round(acc * 100, 2), "%")
print("FPS           :", round(dino_results["fps"], 2))
print("Latency       :", round(dino_results["latency"], 5))
print("GPU Memory GB :", round(dino_results["gpu_memory"], 2))